# Notebook 01: Zero-to-Hero Foundations with Synthetic Data

## Why this notebook exists
**Definition:** Feature selection is the process of choosing the most useful subset of input variables for model training.

**Theory:** In high-dimensional spaces, distance metrics degrade, redundant signals increase variance, and optimization becomes harder. This is the curse of dimensionality.

**Mathematical intuition:** For many estimators, variance of estimated parameters grows with feature count when sample size is fixed. More noisy dimensions increase overfitting risk.

**Real-world examples:**
- Credit scoring with hundreds of engineered attributes.
- Sensor monitoring where many channels are weakly informative.
- Genomics with thousands of features and limited observations.

We start with fully controlled synthetic data so we know ground truth (informative vs redundant vs noise).


## Learning outcomes
By the end of this notebook you will:
1. Build 100 / 500 / 1000+ feature datasets.
2. Observe how accuracy, train time, and overfitting behavior change with dimensionality.
3. Understand why progressive feature-selection funnels are required in production.


In [ ]:
import warnings
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from src.synthetic_generator import (
    compute_informativeness,
    generate_noise_scale_experiment,
    generate_synthetic_dataset,
)

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')

FIG_DIR = Path('outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Experiment 1: What changes when features increase from 100 to 1000?

**Visual explanation:** We keep the number of informative features relatively small while increasing total dimensions.

**Code explanation:** For each feature width, we train the same model and measure generalization gap and runtime.


In [ ]:
settings = [
    {'label': '100 features', 'n_features': 100, 'n_informative': 20, 'n_redundant': 15, 'n_repeated': 5},
    {'label': '500 features', 'n_features': 500, 'n_informative': 25, 'n_redundant': 25, 'n_repeated': 10},
    {'label': '1000 features', 'n_features': 1000, 'n_informative': 30, 'n_redundant': 30, 'n_repeated': 10},
]

rows = []
metadata_snapshots = {}

for cfg in settings:
    X, y, meta = generate_synthetic_dataset(
        n_samples=4000,
        n_features=cfg['n_features'],
        n_informative=cfg['n_informative'],
        n_redundant=cfg['n_redundant'],
        n_repeated=cfg['n_repeated'],
        noise_level=0.25,
        random_state=42,
    )
    metadata_snapshots[cfg['label']] = meta

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    model = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)

    rows.append(
        {
            'scenario': cfg['label'],
            'n_features': cfg['n_features'],
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'generalization_gap': train_acc - test_acc,
            'train_time_sec': train_time,
        }
    )

dimensionality_df = pd.DataFrame(rows)
dimensionality_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

sns.barplot(data=dimensionality_df, x='scenario', y='test_accuracy', ax=axes[0], color='steelblue')
axes[0].set_title('Test Accuracy vs Feature Count')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=dimensionality_df, x='scenario', y='generalization_gap', ax=axes[1], color='firebrick')
axes[1].set_title('Overfitting Gap (Train - Test)')
axes[1].tick_params(axis='x', rotation=15)

sns.barplot(data=dimensionality_df, x='scenario', y='train_time_sec', ax=axes[2], color='darkgreen')
axes[2].set_title('Training Cost vs Feature Count')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(FIG_DIR / 'synthetic_dimensionality_tradeoffs.png', dpi=300, bbox_inches='tight')
plt.show()


## Experiment 2: Noise scaling and feature quality

**Definition:** Noise features are variables that do not carry stable signal about the target.

**Theory:** As observation noise rises, separability between informative and noisy feature groups collapses.

**Mathematical intuition:** F-statistics shrink when between-class variance decreases relative to within-class variance.


In [ ]:
noise_df = generate_noise_scale_experiment(
    noise_levels=[0.0, 0.1, 0.25, 0.5, 1.0, 2.0],
    n_samples=1200,
    n_features=500,
    n_informative=25,
    random_state=42,
)

plt.figure(figsize=(10, 5))
sns.lineplot(data=noise_df, x='noise_level', y='mean_f_stat', hue='feature_type', marker='o')
plt.title('Signal Collapse Under Increasing Noise')
plt.xlabel('Noise Standard Deviation')
plt.ylabel('Mean ANOVA F-statistic')
plt.tight_layout()
plt.savefig(FIG_DIR / 'synthetic_noise_scale_experiment.png', dpi=300, bbox_inches='tight')
plt.show()

noise_df.head(12)


## Interpretation and practical implications

- Test quality does not improve linearly with more columns.
- Training cost rises with dimensionality, and overfitting risk often rises too.
- Controlled synthetic data confirms why production systems require feature filtering and ranking.

In the next notebook, we switch to a real high-dimensional dataset (ISOLET, 613 features).
